# VAR Dataset Construction

Builds `var_dataset.csv` — a quarterly time series for two VAR systems:

| System | Sample | Variables |
|--------|--------|-----------|
| System 1 (Nominal Bond) | ~1961Q3 – 2025Q4 | `rtb, xr, xb, y_nom, dp, spr` |
| System 2 (TIPS) | ~2003Q1 – 2025Q4 | `rtb, xr, xtips, dp, y_real` |

All variables in **quarterly decimal units** (not annualized, not percent).

| File | Content |
|------|---------|
| `ie_data.xls` | Shiller S&P 500 price (P), dividend (D), Real Total Return Price (RTRP) |
| `TB3MS.csv` | 3-month T-bill rate |
| `CPIAUCSL.csv` | Consumer Price Index |
| `feds200628.csv` | SVENY10 + NSS parameters (nominal bond returns, yield spread) |
| `feds200805.csv` | TIPSY10 + NSS parameters — **add to enable System 2** |

> **Note on `xr`:** Uses Shiller S&P 500 RTRP as a proxy for the CRSP VW market portfolio. For exact CCV replication, replace with Ken French monthly factors (Mkt-RF + RF from mba.tuck.dartmouth.edu).

In [42]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("Returns and state data")

for fname in ["feds200628.csv", "TB3MS.csv", "CPIAUCSL.csv", "ie_data.xls"]:
    status = "OK" if (DATA_DIR / fname).exists() else "MISSING"
    print(f"  [{status}] {fname}")
tips_path = DATA_DIR / "feds200805.csv"
print(f"  [{'OK' if tips_path.exists() else 'MISSING — System 2 will be NaN'}] feds200805.csv")

  [OK] feds200628.csv
  [OK] TB3MS.csv
  [OK] CPIAUCSL.csv
  [OK] ie_data.xls
  [OK] feds200805.csv


## 1. Shiller S&P 500 Data (`ie_data.xls`)

Column layout at header row 7 (0-indexed):
- **Col 1 — P**: nominal S&P 500 price
- **Col 2 — D**: trailing 12-month annual dividend (= D₁₂, used directly for `dp`)
- **Col 9 — RTRP**: Real Total Return Price — cumulative real total return index (proxy for CRSP VW market)

Dates are stored as floats (e.g., `1990.01` = January 1990).

In [43]:
shiller_raw = pd.read_excel(
    DATA_DIR / "ie_data.xls",
    sheet_name="Data",
    header=7,          # row 7 (0-indexed) contains column names
    engine="xlrd",
)

def shiller_float_to_date(d):
    """Convert Shiller float date (1990.01 = Jan 1990) to Timestamp."""
    year = int(d)
    month = round((d - year) * 100)
    month = max(1, min(12, month if month > 0 else 1))
    return pd.Timestamp(year=year, month=month, day=1)

date_raw = pd.to_numeric(shiller_raw.iloc[:, 0], errors="coerce")
valid = date_raw.notna()
shiller = shiller_raw.loc[valid].copy()
shiller.index = date_raw[valid].apply(shiller_float_to_date)
shiller.index = shiller.index.to_period("M").to_timestamp("M")  # month-end
shiller.index.name = "Date"

# Use positional indexing — robust against pandas deduplication of duplicate column names
P    = pd.to_numeric(shiller.iloc[:, 1], errors="coerce")  # nominal price
D    = pd.to_numeric(shiller.iloc[:, 2], errors="coerce")  # trailing 12-month annual dividend
RTRP = pd.to_numeric(shiller.iloc[:, 9], errors="coerce")  # real total return price

shiller_monthly = pd.DataFrame({"P": P, "D": D, "RTRP": RTRP})
shiller_monthly.dropna(how="any", inplace=True)

print(f"Shiller: {shiller_monthly.index[0].date()} → {shiller_monthly.index[-1].date()}  ({len(shiller_monthly)} months)")
shiller_monthly.tail(3)

Shiller: 1871-01-31 → 2025-12-31  (1860 months)


,P,D,RTRP
Date,,,
2025-10-31,6735.691739,78.62672,4.542652e+06
2025-11-30,6740.885789,78.77344,4.565885e+06
2025-12-31,6853.025455,78.92016,4.647271e+06


## 2. 3-Month T-Bill Rate and CPI (`TB3MS.csv`, `CPIAUCSL.csv`)

- **TB3MS**: 3-month Treasury bill rate, % per annum (FRED). Used for `r_bill` (the risk-free return).
- **CPIAUCSL**: Consumer Price Index, all urban consumers, seasonally adjusted (FRED). Used for quarterly log inflation `pi_q`.

In [44]:
tbill = pd.read_csv(
    DATA_DIR / "TB3MS.csv",
    index_col="observation_date",
    parse_dates=True,
)
tbill.index = tbill.index.to_period("M").to_timestamp("M")

cpi = pd.read_csv(
    DATA_DIR / "CPIAUCSL.csv",
    index_col="observation_date",
    parse_dates=True,
)
cpi.index = cpi.index.to_period("M").to_timestamp("M")

print(f"TB3MS: {tbill.index[0].date()} → {tbill.index[-1].date()}  ({len(tbill)} months)")
print(f"CPI:   {cpi.index[0].date()} → {cpi.index[-1].date()}  ({len(cpi)} months)")

TB3MS: 1934-01-31 → 2026-01-31  (1105 months)
CPI:   1947-01-31 → 2026-01-31  (949 months)


## 3. GSW Nominal Yield Curve (`feds200628.csv`)

Gurkaynak, Sack & Wright (2006) Nelson-Siegel-Svensson fitted curve. Daily data.

- **SVENY10**: 10-year zero-coupon nominal yield (% p.a., continuously compounded)
- **BETA0–BETA3, TAU1, TAU2**: NSS shape parameters, used to evaluate the yield at **9.75 years**

The 9.75-year yield is needed for the quarterly bond return: a 10-year bond bought at end of quarter _t−1_ is sold one quarter later when it has 9.75 years remaining.

In [45]:
gsw_cols = ["SVENY10", "BETA0", "BETA1", "BETA2", "BETA3", "TAU1", "TAU2"]
gsw_raw = pd.read_csv(
    DATA_DIR / "feds200628.csv",
    skiprows=9,
    index_col="Date",
    parse_dates=True,
    na_values="NA",
)[gsw_cols]

gsw_raw.replace(-999.99, np.nan, inplace=True)  # -999.99 is a null sentinel in the GSW file

print(f"GSW daily: {gsw_raw.index[0].date()} → {gsw_raw.index[-1].date()}  ({len(gsw_raw)} trading days)")
gsw_raw.tail(3)

GSW daily: 1961-06-14 → 2026-02-13  (16873 trading days)


,SVENY10,BETA0,BETA1,BETA2,BETA3,TAU1,TAU2
Date,,,,,,,
2026-02-11,4.2615,2.295773,1.394979,-0.000010,8.990454,1.341698,17.386780
2026-02-12,4.1776,2.186942,1.526926,0.000316,9.012959,1.304103,17.256963
2026-02-13,4.1291,2.175592,1.586217,-0.000109,9.032146,1.116186,17.547154


## 4. NSS Yield Formula + Quarterly Resampling

All series are resampled to **end-of-quarter** dates:  
Q1 = Mar 31, Q2 = Jun 30, Q3 = Sep 30, Q4 = Dec 31.

The Nelson-Siegel-Svensson formula evaluates a yield at any maturity $n$ from the fitted parameters:

$$y(n) = \beta_0 + \beta_1 \frac{1-e^{-n/\tau_1}}{n/\tau_1} + \beta_2\!\left(\frac{1-e^{-n/\tau_1}}{n/\tau_1} - e^{-n/\tau_1}\right) + \beta_3\!\left(\frac{1-e^{-n/\tau_2}}{n/\tau_2} - e^{-n/\tau_2}\right)$$

In [46]:
def nss_yield(n, row):
    """Nelson-Siegel-Svensson yield (% p.a.) at maturity n years."""
    b0, b1, b2, b3 = row["BETA0"], row["BETA1"], row["BETA2"], row["BETA3"]
    t1, t2 = row["TAU1"], row["TAU2"]
    if any(pd.isna(v) for v in [b0, b1, b2, b3, t1, t2]):
        return np.nan
    x1 = n / t1
    term1 = (1 - np.exp(-x1)) / x1
    term2 = term1 - np.exp(-x1)
    x2 = n / t2
    term3 = (1 - np.exp(-x2)) / x2 - np.exp(-x2)
    return b0 + b1 * term1 + b2 * term2 + b3 * term3

# Resample to quarter-end (QE-DEC: quarters end in Dec, Mar, Jun, Sep)
gsw_q = gsw_raw.resample("QE-DEC").last()
gsw_q["y975"] = gsw_q.apply(lambda row: nss_yield(9.75, row), axis=1)

tbill_q = tbill.resample("QE-DEC").last()
cpi_q   = cpi.resample("QE-DEC").last()

P_q    = shiller_monthly["P"].resample("QE-DEC").last()
D_q    = shiller_monthly["D"].resample("QE-DEC").last()
RTRP_q = shiller_monthly["RTRP"].resample("QE-DEC").last()

print("Quarterly observations after resampling:")
for name, s in [("GSW", gsw_q["SVENY10"]), ("TB3MS", tbill_q["TB3MS"]),
                ("CPI", cpi_q["CPIAUCSL"]), ("Shiller RTRP", RTRP_q)]:
    nobs = s.notna().sum()
    print(f"  {name:14s}: {nobs} quarters  {s.first_valid_index().date()} → {s.last_valid_index().date()}")

Quarterly observations after resampling:
  GSW           : 219 quarters  1971-09-30 → 2026-03-31
  TB3MS         : 369 quarters  1934-03-31 → 2026-03-31
  CPI           : 317 quarters  1947-03-31 → 2026-03-31
  Shiller RTRP  : 620 quarters  1871-03-31 → 2025-12-31


## 5. TIPS Yield Curve (`feds200805.csv`) — System 2

GSW real (TIPS) yield curve. Same NSS structure as the nominal curve.

**To enable System 2:** add `feds200805.csv` to the `Returns and state data/` folder.  
Download from: `https://www.federalreserve.gov/data/yield-curve-tables/feds200805.csv`

Expected columns (verify after downloading): `TIPSY10`, `BETA0`, `BETA1`, `BETA2`, `BETA3`, `TAU1`, `TAU2`  
Until the file is present, `xtips` and `y_real` remain `NaN`.

In [47]:
tips_path = DATA_DIR / "feds200805.csv"

if tips_path.exists():
    # feds200805.csv has 18 metadata rows before the header (vs 9 for feds200628.csv)
    tips_cols = ["TIPSY10", "BETA0", "BETA1", "BETA2", "BETA3", "TAU1", "TAU2"]
    tips_raw = pd.read_csv(
        tips_path, skiprows=18, index_col="Date", parse_dates=True, na_values="NA"
    )[tips_cols]
    tips_raw.replace(-999.99, np.nan, inplace=True)

    tips_q = tips_raw.resample("QE-DEC").last()
    tips_q["y975_real"] = tips_q.apply(lambda row: nss_yield(9.75, row), axis=1)

    print(f"TIPS: {tips_raw.index[0].date()} → {tips_raw.index[-1].date()}  ({len(tips_q)} quarters)")
    print(f"TIPSY10 available from: {tips_raw['TIPSY10'].first_valid_index().date()}")
    tips_q[["TIPSY10", "y975_real"]].dropna().tail(3)
else:
    tips_q = pd.DataFrame(columns=["TIPSY10", "y975_real"])
    print("feds200805.csv not found — xtips and y_real will be NaN (System 2 disabled).")

TIPS: 1999-01-04 → 2026-02-13  (109 quarters)
TIPSY10 available from: 1999-01-04


## 6. Variable Construction

**Timing convention:** row dated end-of-quarter _t_ contains returns earned **during** quarter _t_ and state variable levels **observed at** end of quarter _t_.

| Symbol | Formula | Timing |
|--------|---------|--------|
| `r_bill` | `TB3MS_{t-1} / 400` | Rate **set** at start of quarter (end of _t−1_) |
| `pi_q` | `log(CPI_t / CPI_{t-1})` | Log inflation **during** quarter _t_ |
| `rtb` | `r_bill − pi_q` | Ex post real bill return |
| `xr` | `log(RTRP_t / RTRP_{t-1}) − rtb` | Excess stock return (real stock − real bill) |
| `xb` | `−9.75·y975_t/100 + 10·Y10_{t-1}/100 − r_bill` | Excess nominal bond return |
| `y_nom` | `Y10_t / 400` | 10-year nominal yield (SVENY10) |
| `dp` | `log(D_t / P_t)` | Log dividend-price ratio (D is trailing 12-month) |
| `spr` | `(Y10_t − TB3MS_t) / 400` | Yield spread |
| `xtips` | `−9.75·y975_real_t/100 + 10·TIPSY10_{t-1}/100 − r_bill` | Excess TIPS bond return |
| `y_real` | `TIPSY10_t / 400` | Real long yield level |

In [48]:
# Align all quarterly series on a shared index
raw = pd.DataFrame({
    "TB3MS": tbill_q["TB3MS"],
    "CPI":   cpi_q["CPIAUCSL"],
    "Y10":   gsw_q["SVENY10"],
    "y975":  gsw_q["y975"],
    "P":     P_q,
    "D":     D_q,
    "RTRP":  RTRP_q,
})

# Merge TIPS columns (left join → NaN where file unavailable or pre-TIPS era)
if not tips_q.empty:
    raw = raw.join(tips_q[["TIPSY10", "y975_real"]], how="left")
else:
    raw["TIPSY10"]   = np.nan
    raw["y975_real"] = np.nan

# --- Components ---
# T-bill return earned during quarter t = rate SET at end of t-1
r_bill = raw["TB3MS"].shift(1) / 400

# Quarterly log CPI inflation during quarter t
pi_q = np.log(raw["CPI"] / raw["CPI"].shift(1))

# --- System 1: Nominal Bond ---
rtb   = r_bill - pi_q
xr    = np.log(raw["RTRP"] / raw["RTRP"].shift(1)) - rtb
xb    = (-9.75 * raw["y975"] / 100 + 10.0 * raw["Y10"].shift(1) / 100) - r_bill
y_nom = raw["Y10"] / 400   # 10-year nominal yield (SVENY10/400)
dp    = np.log(raw["D"] / raw["P"])
spr   = (raw["Y10"] - raw["TB3MS"]) / 400

# --- System 2: TIPS (NaN if feds200805.csv absent) ---
xtips  = (-9.75 * raw["y975_real"] / 100 + 10.0 * raw["TIPSY10"].shift(1) / 100) - r_bill
y_real = raw["TIPSY10"] / 400

print("Variable construction complete. Non-NaN observation counts:")
for name, s in [("rtb", rtb), ("xr", xr), ("xb", xb), ("y_nom", y_nom),
                ("dp", dp), ("spr", spr), ("xtips", xtips), ("y_real", y_real)]:
    n = s.notna().sum()
    fv = s.first_valid_index()
    lv = s.last_valid_index()
    print(f"  {name:7s}: {n:4d} obs   {fv.date() if fv else 'N/A'} → {lv.date() if lv else 'N/A'}")

Variable construction complete. Non-NaN observation counts:
  rtb    :  316 obs   1947-06-30 → 2026-03-31
  xr     :  315 obs   1947-06-30 → 2025-12-31
  xb     :  185 obs   1980-03-31 → 2026-03-31
  y_nom  :  369 obs   1934-03-31 → 2026-03-31
  dp     :  620 obs   1871-03-31 → 2025-12-31
  spr    :  219 obs   1971-09-30 → 2026-03-31
  xtips  :   89 obs   2004-03-31 → 2026-03-31
  y_real :  109 obs   1999-03-31 → 2026-03-31


## 7. Assemble and Save `var_dataset.csv`

Output columns: `date, rtb, xr, xb, y_nom, dp, spr, xtips, y_real`

- **System 1** sub-sample: `df[["rtb","xr","xb","y_nom","dp","spr"]].dropna()` → starts ~1961Q3
- **System 2** sub-sample: `df[["rtb","xr","xtips","dp","y_real"]].dropna()` → starts ~2003Q1 (once TIPS file added)

In [49]:
var_dataset = pd.DataFrame({
    "rtb":    rtb,
    "xr":     xr,
    "xb":     xb,
    "y_nom":  y_nom,
    "dp":     dp,
    "spr":    spr,
    "xtips":  xtips,
    "y_real": y_real,
}, index=raw.index)

var_dataset.index.name = "date"
var_dataset = var_dataset.dropna(how="all")  # drop rows with no data at all

# --- Diagnostics ---
sys1 = var_dataset[["rtb", "xr", "xb", "y_nom", "dp", "spr"]].dropna()
sys2 = var_dataset[["rtb", "xr", "xtips", "dp", "y_real"]].dropna()

print(f"Full grid:  {var_dataset.index[0].date()} → {var_dataset.index[-1].date()}  ({len(var_dataset)} rows)")
print(f"System 1:   {len(sys1)} clean quarters   {sys1.index[0].date()} → {sys1.index[-1].date()}")
if len(sys2) > 0:
    print(f"System 2:   {len(sys2)} clean quarters   {sys2.index[0].date()} → {sys2.index[-1].date()}")
else:
    print("System 2:   0 quarters (add feds200805.csv to enable)")

display(var_dataset[var_dataset[["rtb","xr","xb","y_nom","dp","spr"]].notna().all(axis=1)].head(5))
display(var_dataset.tail(5))
display(var_dataset.describe().round(5))

# Save
out_path = DATA_DIR / "var_dataset.csv"
var_dataset.to_csv(out_path)
print(f"\nSaved → {out_path}  ({out_path.stat().st_size / 1024:.1f} KB)")

Full grid:  1871-03-31 → 2026-03-31  (621 rows)
System 1:   184 clean quarters   1980-03-31 → 2025-12-31
System 2:   88 clean quarters   2004-03-31 → 2025-12-31


,rtb,xr,xb,y_nom,dp,spr,xtips,y_real
date,,,,,,,,
1980-03-31,-0.010670,-0.048856,-0.193870,0.038000,-2.893241,-0.008216,NaN,NaN
1980-06-30,0.008478,0.063506,0.201221,0.017675,-2.959739,0.006814,NaN,NaN
1980-09-30,0.000848,0.094552,-0.159210,0.025675,-3.036884,0.003113,NaN,NaN
1980-12-31,-0.003687,0.042025,-0.029155,0.038725,-3.076025,-0.009121,NaN,NaN
1981-03-31,0.013581,-0.029163,-0.106523,0.033400,-3.054482,-0.001297,NaN,NaN


,rtb,xr,xb,y_nom,dp,spr,xtips,y_real
date,,,,,,,,
2025-03-31,0.003831,-0.069748,0.034956,0.010500,-4.312764,0.000143,0.037487,0.004608
2025-06-30,0.005354,0.048492,-0.000800,0.010575,-4.356154,0.000151,-0.013927,0.004871
2025-09-30,0.001871,0.082154,0.010214,0.009800,-4.429557,0.000726,0.008463,0.004567
2025-12-31,0.004307,0.040942,-0.001408,0.008975,-4.464009,0.001667,-0.018764,0.004966
2026-03-31,0.007268,NaN,0.016472,0.008925,NaN,0.001398,0.019613,0.004422


,rtb,xr,xb,y_nom,dp,spr,xtips,y_real
count,316.00000,315.00000,185.00000,369.00000,620.00000,219.00000,89.00000,109.00000
mean,0.00133,0.01746,0.00915,0.00856,-3.26520,0.00388,0.00045,0.00354
std,0.00831,0.07283,0.06123,0.00775,0.47323,0.00358,0.03897,0.00324
min,-0.03289,-0.31681,-0.19387,0.00002,-4.49746,-0.00912,-0.12006,-0.00257
25%,-0.00199,-0.01387,-0.02542,0.00095,-3.50190,0.00139,-0.02131,0.00137
50%,0.00164,0.02682,0.00357,0.00742,-3.17006,0.00411,0.00769,0.00403
75%,0.00619,0.06335,0.04346,0.01300,-2.92698,0.00649,0.02871,0.00554
max,0.03759,0.21806,0.20122,0.03873,-1.97786,0.01033,0.08280,0.01071



Saved → Returns and state data\var_dataset.csv  (50.8 KB)


---\n## VAR Estimation\n\nBoth systems estimated as VAR(1) with constant, OLS equation-by-equation (= MLE under Gaussian errors).\n\nOutput format mirrors **CCV Table 2**:\n- **Φ̂₁**: full coefficient matrix with *t*-statistics (df-corrected, *T* − *k* − 1 per equation)\n- **R²** per equation\n- **Σ̂_v**: innovation matrix — std dev on diagonal, correlations off-diagonal

In [ ]:
from statsmodels.tsa.vector_ar.var_model import VAR

df = pd.read_csv(DATA_DIR / "var_dataset.csv", index_col="date", parse_dates=True)

### System 1: Nominal Bond VAR(1) — CCV Replication\n\n$z_t = (\\text{rtb}_t,\\; \\text{xr}_t,\\; \\text{xb}_t,\\; \\text{y\\_nom}_t,\\; \\text{dp}_t,\\; \\text{spr}_t)'$

In [ ]:
sys1_cols = ["rtb", "xr", "xb", "y_nom", "dp", "spr"]
data1 = df[sys1_cols].dropna()

print(f"System 1: {data1.index[0].date()} → {data1.index[-1].date()}  (T = {len(data1) - 1})\n")
res1 = VAR(data1).fit(maxlags=1, ic=None, trend='c')
print(res1.summary())

### System 2: TIPS VAR(1)\n\n$z_t = (\\text{rtb}_t,\\; \\text{xr}_t,\\; \\text{xtips}_t,\\; \\text{dp}_t,\\; \\text{y\\_real}_t)'$\n\nHeadline question: how does $\\text{Corr}(v^{xr}, v^{xtips})$ compare to $\\text{Corr}(v^{xr}, v^{xb})$ from System 1?

In [ ]:
sys2_cols = ["rtb", "xr", "xtips", "dp", "y_real"]
data2 = df[sys2_cols].dropna()

print(f"System 2: {data2.index[0].date()} → {data2.index[-1].date()}  (T = {len(data2) - 1})\n")
res2 = VAR(data2).fit(maxlags=1, ic=None, trend='c')
print(res2.summary())

In [ ]:
r2_1 = pd.Series(1 - res1.resid.var(0) / res1.endog[1:].var(0), index=sys1_cols)
r2_2 = pd.Series(1 - res2.resid.var(0) / res2.endog[1:].var(0), index=sys2_cols)

print("System 1:\n", r2_1.round(3))
print("\nSystem 2:\n", r2_2.round(3))